In [1]:
# --- RESOURCE MANAGEMENT (be polite on shared server) ---
import os
import torch

# ── CPU Thread Limits ──────────────────────────────────────────────────────────
# Server has 64 CPUs. Claim at most 8 so others can work.
# PyTorch uses two thread pools: intra-op (math inside one op) and
# inter-op (parallelism across independent ops).
CPU_CORES   = 32   # ← adjust this. 4–8 is polite on a 64-core server.
torch.set_num_threads(CPU_CORES)            # intra-op parallelism
torch.set_num_interop_threads(CPU_CORES)    # inter-op parallelism
os.environ["OMP_NUM_THREADS"]  = str(CPU_CORES)  # OpenMP (used by numpy/scipy)
os.environ["MKL_NUM_THREADS"]  = str(CPU_CORES)  # Intel MKL (used by numpy)

# ── GPU Setup ─────────────────────────────────────────────────────────────────
# If GPU becomes available later, set CUDA_VISIBLE_DEVICES to limit which
# GPU(s) this notebook uses. E.g. "0" = first GPU only, "0,1" = two GPUs.
# Leave blank ("") to use all visible GPUs.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # claim 1 GPU when available

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)
import numpy as np
np.random.seed(SEED)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Device          : {DEVICE}")
print(f"CPU threads     : {torch.get_num_threads()} intra / {torch.get_num_interop_threads()} inter")
print(f"OMP/MKL threads : {CPU_CORES}")
if DEVICE == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(f"GPU             : Not available (running on CPU)")
    print(f"  → If you have GPU access, run: ssh <gpu-node> or use your cluster's job scheduler")


Device          : cpu
CPU threads     : 32 intra / 32 inter
OMP/MKL threads : 32
GPU             : Not available (running on CPU)
  → If you have GPU access, run: ssh <gpu-node> or use your cluster's job scheduler


In [2]:
# --- IMPORTS & LOAD CHECKPOINTS ---
import numpy as np, pandas as pd, pickle, torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

# Load saved artifacts
train_df = pd.read_pickle('./checkpoints/train_df.pkl')
val_df   = pd.read_pickle('./checkpoints/val_df.pkl')
test_df  = pd.read_pickle('./checkpoints/test_df.pkl')

with open('./checkpoints/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)
with open('./checkpoints/settings.pkl', 'rb') as f:
    S = pickle.load(f)

embed_matrix = torch.load('./checkpoints/embed_matrix.pt')
pos_weights  = torch.load('./checkpoints/pos_weights.pt')

MAX_LEN       = S['MAX_LEN']
BATCH_SIZE    = S['BATCH_SIZE_RNN']
TARGET_LABELS = S['TARGET_LABELS']
EMBEDDING_DIM = S['EMBEDDING_DIM']

print(f"✓ Loaded: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print(f"✓ Vocab: {len(word2idx):,} | Embed: {tuple(embed_matrix.shape)}")


/home/emmy/.conda/envs/mlbio_esm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
✓ Loaded: train=6956, val=987, test=1976
✓ Vocab: 31,756 | Embed: (31756, 100)


/tmp/ipykernel_634109/104470028.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embed_matrix = torch.load('./checkpoints/embed_matrix.pt')
/tmp/ipykernel_634109/1044700

In [3]:
# --- DATASET CLASS ---
from torch.utils.data import Dataset   # ← add this missing import

class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, target_labels, max_len=64,
                 model_type='bert', word2idx=None):
        self.df            = df
        self.tokenizer     = tokenizer
        self.target_labels = target_labels
        self.max_len       = max_len
        self.model_type    = model_type
        self.word2idx      = word2idx
        self.has_labels    = all(l in df.columns for l in target_labels)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        tweet = row['tweet']

        if self.model_type == 'bert':
            enc = self.tokenizer(tweet, max_length=self.max_len,
                                 padding='max_length', truncation=True,
                                 return_tensors='pt')
            out = {
                'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'token_type_ids': enc.get('token_type_ids', torch.zeros(self.max_len, dtype=torch.long)).squeeze(),
            }
        else:  # rnn
            tokens = tweet.split()[:self.max_len]
            ids    = [self.word2idx.get(t, self.word2idx['<UNK>']) for t in tokens]
            pad    = self.word2idx['<PAD>']
            ids   += [pad] * (self.max_len - len(ids))
            out    = {
                'input_ids':      torch.tensor(ids, dtype=torch.long),
                'attention_mask': torch.tensor([1]*len(tokens) + [0]*(self.max_len - len(tokens)), dtype=torch.long),
            }

        if self.has_labels:
            out['labels'] = torch.tensor([row[l] for l in self.target_labels], dtype=torch.float32)
        return out

# No bert_tokenizer needed here — BiLSTM uses model_type='rnn'
print("✓ TweetDataset class ready")

✓ TweetDataset class ready


In [4]:
# --- RNN DATALOADERS ---
train_ds = TweetDataset(train_df, None, TARGET_LABELS, MAX_LEN, 'rnn', word2idx)
val_ds   = TweetDataset(val_df,   None, TARGET_LABELS, MAX_LEN, 'rnn', word2idx)
test_ds  = TweetDataset(test_df,  None, TARGET_LABELS, MAX_LEN, 'rnn', word2idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"✓ Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")


✓ Train: 218 batches | Val: 31 | Test: 62


In [5]:
# --- BILSTM MODEL ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=128, num_labels=12,
                 num_layers=2, dropout=0.4, freeze_embeddings=False):
        super().__init__()
        self.embedding  = nn.Embedding.from_pretrained(embed_matrix, freeze=freeze_embeddings, padding_idx=0)
        _, emb_dim      = embed_matrix.shape
        self.lstm       = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                                  batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
                                  bidirectional=True)
        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.fc         = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, attention_mask=None, **kwargs):
        x = self.dropout(self.embedding(input_ids))
        x, _ = self.lstm(x)
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        else:
            x = x.mean(1)
        return self.fc(self.dropout(self.layer_norm(x)))

model = BiLSTMClassifier(embed_matrix, hidden_dim=128, num_labels=len(TARGET_LABELS)).to(DEVICE)
print(f"✓ BiLSTM params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


✓ BiLSTM params: 3,809,980


In [6]:
# --- 8. TRAINING UTILITIES & HELPER FUNCTIONS ---

def get_loss_fn(pos_weights, device):
    return nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))


def compute_macro_f1(y_true, y_pred, threshold=0.5):
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    y_pred_binary = (y_pred > threshold).astype(int)
    return f1_score(y_true, y_pred_binary, average='macro', zero_division=0)


class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience   = patience
        self.delta      = delta
        self.best_score = None
        self.counter    = 0
        self.best_model = None

    def __call__(self, val_score, model):
        if self.best_score is None or val_score > self.best_score + self.delta:
            self.best_score = val_score
            self.counter    = 0
            self.best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience


def train_epoch(model, dataloader, optimizer, loss_fn, device, scheduler=None, clip_grad=1.0):
    """
    FIX #3: scheduler is now stepped per BATCH (not per epoch).
    FIX +: gradient clipping prevents exploding gradients.
    """
    model.train()
    total_loss = 0.0

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss   = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        if clip_grad:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)  # gradient clip
        optimizer.step()
        if scheduler is not None:
            scheduler.step()  # FIX #3: step per batch, not per epoch

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, loss_fn, device, threshold=0.5):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss   = loss_fn(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    macro_f1   = compute_macro_f1(all_labels, all_preds, threshold)

    return {'loss': total_loss / len(dataloader), 'macro_f1': macro_f1,
            'preds': all_preds, 'labels': all_labels}


def predict_test(model, dataloader, device):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(all_preds, axis=0)


def run_training(model, train_loader, val_loader, optimizer, scheduler,
                 loss_fn, device, epochs, patience=5, model_name="Model"):
    """Generic training loop used for any model."""
    early_stopping = EarlyStopping(patience=patience, delta=0.001)
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_val_f1 = 0.0

    print(f"\n{'='*70}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*70}\n")

    for epoch in range(epochs):
        train_loss  = train_epoch(model, train_loader, optimizer, loss_fn, device, scheduler)
        val_results = evaluate(model, val_loader, loss_fn, device)
        val_loss    = val_results['loss']
        val_f1      = val_results['macro_f1']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        marker = " ★ NEW BEST" if val_f1 > best_val_f1 else ""
        best_val_f1 = max(best_val_f1, val_f1)
        print(f"Epoch {epoch+1:>2}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val F1: {val_f1:.4f}{marker}")

        if early_stopping(val_f1, model):
            print(f"\n⚠ Early stopping at epoch {epoch+1}")
            break

    if early_stopping.best_model:
        model.load_state_dict(early_stopping.best_model)
        print(f"\n✓ Loaded best weights (Val F1: {early_stopping.best_score:.4f})")

    return history, early_stopping.best_score


def make_submission(model, test_loader, test_df, device, filename, threshold=0.5):
    """Generate predictions and save Kaggle submission CSV."""
    preds = predict_test(model, test_loader, device)
    binary = (preds > threshold).astype(int)
    sub = pd.DataFrame({'index': test_df['ID'].values})
    for i, label in enumerate(TARGET_LABELS):
        sub[label] = binary[:, i]
    sub.to_csv(filename, index=False)
    print(f"✓ Submission saved: {filename}  shape={sub.shape}")
    return sub


# Initialize loss function
loss_fn = get_loss_fn(pos_weights, DEVICE)
print("✓ All training utilities defined (scheduler-per-batch, grad clip, early stopping)")
print(f"  Device: {DEVICE}")


✓ All training utilities defined (scheduler-per-batch, grad clip, early stopping)
  Device: cpu


In [7]:
# --- TRAIN BILSTM ---
EPOCHS = 20
LR     = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn   = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(DEVICE))
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

history, best_f1 = run_training(model, train_loader, val_loader,
    optimizer, scheduler, loss_fn, DEVICE, EPOCHS, patience=5, model_name="BiLSTM")

torch.save(model.state_dict(), './checkpoints/bilstm_best.pt')
print(f"\n✓ Model saved to ./checkpoints/bilstm_best.pt")
print(f"✓ Best Val F1: {best_f1:.4f}")



TRAINING: BiLSTM

Epoch  1/20 | Train Loss: 0.9602 | Val Loss: 0.8246 | Val F1: 0.2575 ★ NEW BEST
Epoch  2/20 | Train Loss: 0.8183 | Val Loss: 0.7404 | Val F1: 0.3511 ★ NEW BEST
Epoch  3/20 | Train Loss: 0.7066 | Val Loss: 0.6755 | Val F1: 0.4251 ★ NEW BEST
Epoch  4/20 | Train Loss: 0.5879 | Val Loss: 0.6369 | Val F1: 0.4551 ★ NEW BEST
Epoch  5/20 | Train Loss: 0.4702 | Val Loss: 0.6723 | Val F1: 0.5210 ★ NEW BEST
Epoch  6/20 | Train Loss: 0.3695 | Val Loss: 0.6508 | Val F1: 0.4711
Epoch  7/20 | Train Loss: 0.2874 | Val Loss: 0.7406 | Val F1: 0.5386 ★ NEW BEST
Epoch  8/20 | Train Loss: 0.2253 | Val Loss: 0.8365 | Val F1: 0.5371
Epoch  9/20 | Train Loss: 0.1858 | Val Loss: 0.8862 | Val F1: 0.5455 ★ NEW BEST
Epoch 10/20 | Train Loss: 0.1538 | Val Loss: 1.0555 | Val F1: 0.5351
Epoch 11/20 | Train Loss: 0.1335 | Val Loss: 1.0534 | Val F1: 0.5379
Epoch 12/20 | Train Loss: 0.1159 | Val Loss: 1.1532 | Val F1: 0.5502 ★ NEW BEST
Epoch 13/20 | Train Loss: 0.1022 | Val Loss: 1.2007 | Val F1: 0.5

In [8]:
# --- SUBMISSION & PER-LABEL ANALYSIS ---
preds  = predict_test(model, test_loader, DEVICE)
binary = (preds > 0.5).astype(int)

sub = pd.DataFrame({'index': test_df['ID'].values})
for i, label in enumerate(TARGET_LABELS):
    sub[label] = binary[:, i]
sub.to_csv('./submission_bilstm.csv', index=False)
print("✓ Submission saved: submission_bilstm.csv")

# Per-label F1 on val set
val_res = evaluate(model, val_loader, loss_fn, DEVICE)
val_bin = (val_res['preds'] > 0.5).astype(int)
print(f"\nVal Macro F1: {val_res['macro_f1']:.4f}\n")
print(f"{'Label':<15} {'F1':>6}")
print("-" * 25)
for i, label in enumerate(TARGET_LABELS):
    lf1 = f1_score(val_res['labels'][:, i], val_bin[:, i], zero_division=0)
    print(f"{label:<15} {lf1:>6.4f}")

# Save history for comparison notebook
import pickle
with open('./checkpoints/history_bilstm.pkl', 'wb') as f:
    pickle.dump({'history': history, 'best_f1': best_f1, 'val_results': val_res}, f)


✓ Submission saved: submission_bilstm.csv

Val Macro F1: 0.5589

Label               F1
-------------------------
ineffective     0.6105
unnecessary     0.4318
pharma          0.5891
rushed          0.6200
side-effect     0.8103
mandatory       0.6919
country         0.4390
ingredients     0.6000
political       0.3423
none            0.4528
conspiracy      0.4528
religious       0.6667


In [ ]:
# --- PER-LABEL THRESHOLD TUNING (free F1 improvement) ---
from sklearn.metrics import f1_score
import numpy as np

val_res = evaluate(model, val_loader, loss_fn, DEVICE)
probs   = val_res['preds']    # shape: (987, 12)
labels  = val_res['labels']

best_thresholds = []
print(f"{'Label':<15} {'Best Thresh':>12} {'F1 at 0.5':>10} {'F1 tuned':>10}")
print("-" * 52)
for i, label in enumerate(TARGET_LABELS):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.05):
        preds_t = (probs[:, i] > t).astype(int)
        f = f1_score(labels[:, i], preds_t, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    default_f1 = f1_score(labels[:, i], (probs[:, i] > 0.5).astype(int), zero_division=0)
    best_thresholds.append(best_t)
    print(f"{label:<15} {best_t:>12.2f} {default_f1:>10.4f} {best_f1:>10.4f}")

# Apply tuned thresholds to test predictions and save
test_preds = predict_test(model, test_loader, DEVICE)
binary_tuned = np.stack([
    (test_preds[:, i] > best_thresholds[i]).astype(int)
    for i in range(len(TARGET_LABELS))
], axis=1)
sub_tuned = pd.DataFrame({'index': test_df['ID'].values})
for i, label in enumerate(TARGET_LABELS):
    sub_tuned[label] = binary_tuned[:, i]
sub_tuned.to_csv('./submission_bilstm_tuned.csv', index=False)
print(f"\n✓ Tuned submission saved: submission_bilstm_tuned.csv")

import pickle
with open('./checkpoints/thresholds_bilstm.pkl', 'wb') as f:
    pickle.dump(best_thresholds, f)
